# 33. Communicating findings
A useful report connects the prediction problem, evidence, and decision. Explain the baseline, appropriate evaluation split, metric units, uncertainty, and remaining weaknesses. Avoid claiming that a tiny score improvement is meaningful without checking its variability and operational effect.

Write six short sections: problem and prediction time; data and representativeness; evaluation design; baseline and candidate results; errors and subgroup checks; recommendation and limitations. Distinguish observations from hypotheses. For small subgroups report counts and uncertainty. Name what the model cannot establish and what additional evidence is needed.

## Metric uncertainty, OOF predictions, and a permutation test
Different uncertainty questions need different procedures. A paired bootstrap of fixed held-out (actual, prediction) pairs estimates sampling variability of a metric conditional on a fixed fitted model; it does not include retraining or tuning uncertainty. Resample entities or time blocks instead of rows when observations are dependent. cross_val_predict produces out-of-fold predictions, but a metric on pooled predictions can differ from a mean fold score, especially for non-decomposable metrics. It also does not remove selection bias if its settings were chosen using the same data. permutation_test_score asks whether the scoring procedure outperforms a label-permutation null under exchangeability; it is not a comparison between two tuned models or a causal test.

In [ ]:
import numpy as np
from sklearn.datasets import make_regression, load_wine
from sklearn.model_selection import train_test_split, cross_val_predict, permutation_test_score, StratifiedKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import mean_absolute_error
X, y = make_regression(n_samples=180, n_features=4, noise=15, random_state=42)
Xt, Xv, yt, yv = train_test_split(X, y, random_state=42)
pred = LinearRegression().fit(Xt, yt).predict(Xv) # Fixed held-out predictions for one prechosen model.
rng = np.random.default_rng(42)
errors = []
for _ in range(300):
    index = rng.integers(0, len(yv), size=len(yv)) # Resample paired rows together.
    errors.append(mean_absolute_error(yv[index], pred[index]))
print("Held-out MAE:", round(mean_absolute_error(yv, pred), 3))
print("Approximate 95% percentile interval:", np.quantile(errors, [0.025, 0.975]).round(3).tolist()) # Conditional metric uncertainty, not prediction bounds.
W, labels = load_wine(return_X_y=True)
pipe = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
cv = StratifiedKFold(3, shuffle=True, random_state=42)
oof = cross_val_predict(pipe, W, labels, cv=cv, method="predict") # Each row predicted without fitting that row.
score, null_scores, pvalue = permutation_test_score(pipe, W, labels, cv=cv, n_permutations=19, random_state=42, scoring="accuracy") # Small demonstration, coarse p-value resolution.
print("OOF prediction count:", len(oof))
print("Observed score / permutation p-value:", round(score, 3), round(pvalue, 3))
assert len(oof) == len(labels) and len(null_scores) == 19

### Reading the result
The percentile endpoints are an approximate uncertainty interval for mean absolute error under the stated row-bootstrap assumptions. They do not bound individual future targets. With 19 permutations, p-values have resolution 1/20=0.05 using the corrected count; this is a teaching budget, not a powerful significance study. A low p-value does not quantify practical usefulness.

### Transfer exercise
Write a report paragraph that clearly separates a point metric, an approximate conditional metric interval, and a prediction interval. Explain when grouped resampling is necessary.

## Subgroup checks and fairness questions
Aggregate quality can hide unequal error rates. Before comparing groups, define the target, the decision, group membership, the harms of false positives and false negatives, and whether the data represents each group. Report counts and support alongside metrics. Small groups produce noisy estimates; intersectional groups can be especially sparse. Equal recall, equal false-positive rates, equal calibration, and equal selection rates are distinct goals and can conflict. A numerical parity check is not a fairness certificate, and sensitive features can still be proxied by other columns after removal.

In [ ]:
import numpy as np
from sklearn.metrics import confusion_matrix, recall_score, precision_score
actual = np.array([1, 1, 1, 0, 0, 0, 1, 1, 0, 0, 0, 0])
predicted = np.array([1, 0, 1, 1, 0, 0, 1, 0, 0, 0, 0, 0])
groups = np.array(["A"] * 6 + ["B"] * 6) # Illustrative groups, not a claim about real populations.
for group in ["A", "B"]:
    mask = groups == group
    tn, fp, fn, tp = confusion_matrix(actual[mask], predicted[mask], labels=[0, 1]).ravel() # Explicit labels preserve matrix shape.
    print(group, "n", int(mask.sum()), "positives", int(actual[mask].sum()), "recall", round(recall_score(actual[mask], predicted[mask], zero_division=0), 3), "FPR", round(fp / (fp + tn), 3))
assert len(groups) == len(actual) == len(predicted)

### Reading the result
The six-row groups are too small for strong conclusions. The example demonstrates reporting counts and denominators. A zero denominator needs explicit handling and should not be hidden as a good score. Real fairness analysis also requires context, affected people’s needs, label-quality investigation, and a justified decision policy.

### Transfer exercise
Add subgroup counts and error types to an earlier project report. Name one unmeasured group or harm that your analysis cannot evaluate.

## Your turn
Write a one-page report for an earlier project. Include one finding, one uncertainty, one subgroup check with counts, and one bounded recommendation.

Try the task before opening the solution on the course website.